In [0]:
%run ./00_project_setup

In [0]:
from src.metadata import (
    ColumnProfile,
    DatasetFingerprint,
    DatasetProfile,
    DatasetStatistics,
    MetadataCatalogEntry,
)

print("Implementation 07 metadata domain imports: PASSED")

## Section 02 — Spark Dataset Profiler Validation

This section validates the reusable Spark profiling engine using a controlled
dataset with known row counts, duplicate rows, null values, numeric fields,
string fields, and a date field.

In [0]:
from pyspark.sql import functions as F

from src.metadata import SparkDatasetProfiler


profiler_test_df = (
    spark.createDataFrame(
        [
            (1, "Monday", 10000.0, "2026-07-27"),
            (2, "Tuesday", 10500.0, "2026-07-28"),
            (3, "Wednesday", None, "2026-07-29"),
            (3, "Wednesday", None, "2026-07-29"),
        ],
        [
            "record_id",
            "day_name",
            "released_lines",
            "release_date_text",
        ],
    )
    .withColumn(
        "release_date",
        F.to_date("release_date_text"),
    )
    .drop("release_date_text")
)

metadata_profiler = SparkDatasetProfiler(
    approximate_distinct=False
)

dataset_statistics, column_profiles = metadata_profiler.profile(
    profiler_test_df
)

print("Row count:", dataset_statistics.row_count)
print("Column count:", dataset_statistics.column_count)
print("Duplicate rows:", dataset_statistics.duplicate_row_count)
print("Null cells:", dataset_statistics.null_cell_count)
print("Columns profiled:", len(column_profiles))

assert dataset_statistics.row_count == 4
assert dataset_statistics.column_count == 4
assert dataset_statistics.duplicate_row_count == 1
assert dataset_statistics.null_cell_count == 2
assert len(column_profiles) == 4

print("Implementation 07 Spark dataset profiler: PASSED")

## Dataset Fingerprint Validation  

Validate deterministic schema, content, metadata, and combined dataset  
fingerprints using the profiler test dataset.

In [0]:
from src.metadata import (
    DatasetFingerprint,
    DatasetFingerprintGenerator,
)


fingerprint_generator = DatasetFingerprintGenerator()

fingerprint_metadata = {
    "dataset_key": "metadata_profiler_test",
    "dataset_layer": "TEST",
    "project_name": PROJECT_NAME,
    "environment": ENVIRONMENT,
}


first_fingerprint = fingerprint_generator.generate(
    profiler_test_df,
    metadata=fingerprint_metadata,
    statistics=dataset_statistics,
)

second_fingerprint = fingerprint_generator.generate(
    profiler_test_df,
    metadata=fingerprint_metadata,
    statistics=dataset_statistics,
)


print("Schema hash        :", first_fingerprint.schema_hash)
print("Content hash       :", first_fingerprint.content_hash)
print("Metadata hash      :", first_fingerprint.metadata_hash)
print("Combined hash      :", first_fingerprint.combined_hash)
print("Row count          :", first_fingerprint.row_count)
print("Column count       :", first_fingerprint.column_count)
print("Fingerprint version:", first_fingerprint.fingerprint_version)
print("Algorithm          :", first_fingerprint.algorithm)
print("Generated at UTC   :", first_fingerprint.generated_at_utc)


assert isinstance(
    first_fingerprint,
    DatasetFingerprint,
)

assert first_fingerprint.row_count == 4
assert first_fingerprint.column_count == 4

assert len(first_fingerprint.schema_hash) == 64
assert len(first_fingerprint.content_hash) == 64
assert len(first_fingerprint.metadata_hash) == 64
assert len(first_fingerprint.combined_hash) == 64

assert first_fingerprint.fingerprint_version == "1.0.0"
assert first_fingerprint.algorithm == "SHA-256"

assert (
    first_fingerprint.schema_hash
    == second_fingerprint.schema_hash
)

assert (
    first_fingerprint.content_hash
    == second_fingerprint.content_hash
)

assert (
    first_fingerprint.metadata_hash
    == second_fingerprint.metadata_hash
)

assert (
    first_fingerprint.combined_hash
    == second_fingerprint.combined_hash
)

assert DatasetFingerprintGenerator.fingerprints_match(
    first_fingerprint,
    second_fingerprint,
)

assert (
    first_fingerprint.generated_at_utc
    <= second_fingerprint.generated_at_utc
)

print(
    "Implementation 07 dataset fingerprinting: PASSED"
)